In [ ]:
# librerías importadas
import numpy as np
import pandas as pd
import seaborn as sns
# Módulos de librerías
import matplotlib.pyplot as plt
# Rutinas específicas
from numpy.linalg import norm

## **Análisis de las estrellas de nuestra galaxia en un campo de observación**

El pripeline hace lo siguiente:

`ADQL` contiene el query asociado a nuestra consulta, para ello pedimos las coordenadas asociadas a GAIA y las magnitudes de las tablas de GAIA y WISE. Para diferenciar entonces las tablas definimos a GAIA con la letra g y a WISE con w.

Luego le pedimos hacer el INNER JOIN entre la tabla de GAIA, que tomamos como principal, con la tabla de WISE (secundaria en este caso).

Para las condiciones le decimos que en un círculo de 0.5 grados busque en la región de coordenadas (135.5, 0.5) (asención recta y declinación) las estrellas que estén allí y coincidan además. Incluso que relacione la distancia con el comando `DISTANCE` en 2 arcosegundos.



In [ ]:
%%bash

echo "Pipeline de consulta en VizieR"

ADQL="
SELECT
g.RA_ICRS,
g.DE_ICRS,
g.Gmag,
g.Plx,
g.pmRA,
g.pmDE,
w.W1mag
FROM \"I/355/gaiadr3\" AS g
INNER JOIN \"II/328/allwise\" AS w
  ON 1=CONTAINS(
    POINT('ICRS', g.RA_ICRS, g.DE_ICRS),
    CIRCLE('ICRS', 135.5, 0.5, 0.5))
  AND 1=CONTAINS(
    POINT('ICRS', w.RAJ2000, W.DEJ2000),
    CIRCLE('ICRS', 135.5, 0.5, 0.5)
  )
  AND DISTANCE(
    POINT('ICRS', g.RA_ICRS, g.DE_ICRS),
    POINT('ICRS', w.RAJ2000, w.DEJ2000)
  ) < 2.0/3600.0
"

URL_ADQL=$(echo $ADQL | sed 's/ /+/g')

TAP_URL="https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync?request=doQuery&lang=ADQL&format=csv&query="

echo $TAP_URL$URL_ADQL

wget -q -O mw_stars.csv "$TAP_URL$URL_ADQL"

echo "Datos descargados"

## **Lectura de datos**

Aquí vamos a leer primero los datos que tenemos de neustra consulta que quedaron guardados en nuestro archivo `mw_stars.csv`.

Los datos solicitados fueron la `Gmag`y la `W1mag`, y tienen para cada estrella las coordenadas de Asención Recta y Declinación respectivas con la referencia de J2000.

In [ ]:
df_stars_mw = pd.read_csv('mw_stars.csv').dropna()

df_stars_mw.head(10)

In [ ]:
print(len(df_stars_mw))

## **Estrellas que están en la Vía Láctea**

Para esta región podemos llegar a tener una estrella que esté por fuera de nuestra galaxia, así que le pedimos a VizieR en el TAP que definimos en el script de ADQL que nos trajera la información de la cinemática de las estrellas: movimiento propio y paralaje.

Vamos a ver primero qué estrellas tienen un paralaje sospechoso y luego vamos a ver aquellas que tienen un movimiento propio raro.

El paralaje, llamémoslo $\theta$, en GAIA está medido en miliarcosegundos (mas), lo que quiere decir que los números que obtengamos serán muy pequeños. Veámolos a continuación:

In [ ]:
plt.style.use('dark_background')

plt.hist(df_stars_mw['Plx'], bins=50, color='blue')
plt.title('Paralaje de las estrellas del query')
plt.xlabel(r'Plx $\theta$ (mas)')
plt.ylabel('Número de estrellas')
plt.grid()
plt.show()

Para todas las estrellas con un paralaje $\theta < 0$ vamos a descartalas.

In [ ]:
df_plx = df_stars_mw[df_stars_mw['Plx'] > 0].copy()

print(len(df_plx))

In [ ]:
plt.style.use('dark_background')

plt.hist(df_plx['Plx'], bins=50, color='blue')
plt.title(r'Paralaje de las estrellas del query con $\theta > 0$')
plt.xlabel(r'Plx $\theta$ (mas)')
plt.ylabel('Número de estrellas')
plt.grid()
plt.show()

Ahora vamos a analizar el movimiento propio, ya que los objetos más lejanos, como galaxias, estrellas externas o quásares suelen tener un movimiento propio muy pequeño, o prácticamente nulo.

In [ ]:
plt.scatter(df_plx['pmDE'], df_plx['pmRA'], c='blue', alpha=0.6)
plt.title('Movimiento propio de las estrellas de la muestra')
plt.xlabel(r'$\mu_{DE}$ (mas)')
plt.ylabel(r'$\mu_{RA}$ (mas)')
#plt.grid()
plt.show()

Vamos a aplicar el siguiente filtro, vamos a crear un vector de movimiento propio: 

$$\vec{\mu} = (\mu_\text{RA}, \mu_\text{DE})$$

Con este vector calcularemos su norma $\mu$ y si esta es muy pequeña la descartaremos.

In [ ]:
mus_dec = np.array(df_plx['pmDE'])
mus_ra = np.array(df_plx['pmRA'])

print(len(mus_ra), len(mus_dec))

Para filtrar nuestro cúmulo estelar calculemos el primedio de los movimientos propios del sistema para más o menos mirar donde está centrado nuestra distribución de movimiento.

In [ ]:
mean_mu = np.array([np.mean(mus_ra), np.mean(mus_dec)])

mean_mu

A continuación podemos ver que el promedio no está en el cero, sino en una región cercana a $(-4.4706, -3.6958)$ lo que nos dice que estas estrellas que están en grupo son parte de un cúmulo estelar.

Las estrellas más compactas son del cúmulo de interés, las más alejadas son estrellas locales que no pertenecen al cúmulo del campo profundo, mientras que las estrellas que están muy cerca del $(0,0)$ son objetos muy lejandos. Para determinar quiénes sí son del cúmulo primero vamos a establecer un radio de corte.

In [ ]:
plt.scatter(df_plx['pmRA'], df_plx['pmDE'], c='blue', alpha=0.6)
plt.plot(mean_mu[0], mean_mu[1], 'r+')
plt.title('Movimiento propio de las estrellas de la muestra')
plt.xlabel(r'$\mu_{RA}$ (mas)')
plt.ylabel(r'$\mu_{DE}$ (mas)')
plt.grid(alpha=0.4)
plt.show()

Aquí creamos nuestra matriz de vectores de movimiento proipio.

In [ ]:
mus = np.zeros((len(mus_ra), 2))
mus[:,0], mus[:,1] = mus_ra, mus_dec
mus

El radio de corte se define así:

$$R^2 = (\mu_\text{RA} - \text{RA}_\text{prom})^2 + (\mu_\text{DE} - \delta_\text{prom})^2 $$

La norma de todos ellos se guarda en el siguiente array:

In [ ]:
norm_mus = norm(mus, axis=1)

mask_mus = norm_mus < 20

df_plx['norm_mus'] = norm_mus

In [ ]:
df_mus = df_plx[mask_mus]

len(df_mus)

Ahora sí vemos aquí nuestras estrellas del cúmulo.

In [ ]:
plt.scatter(df_mus['pmRA'], df_mus['pmDE'], c='blue', alpha=0.6)
plt.plot(mean_mu[0], mean_mu[1], 'r+')
plt.title('Estrellas del cúmulo del campo profundo')
plt.xlabel(r'$\mu_{RA}$ (mas)')
plt.ylabel(r'$\mu_{DE}$ (mas)')
plt.grid(alpha=0.4)
plt.show()

Para ya descartar las que son muy lejanas miremos el paralaje nuevamente:

In [ ]:
plt.plot(df_mus['Plx'], 'b.')
plt.ylim(0, 1)

## **Diagrama color-magnitud de las estrellas en la Vía Láctea**

Para nuestro diagrama color magnitud calcularemos el índice de color entre el visible y el infrarrojo:

$$
C = V - I
$$

Donde $C$ representa el índice de color dado por la resta entre las magnitudes de las dos tablas.

In [ ]:
df_stars_mw.keys()

Por simplicidad vamos a ver cuántas estrellas hay por cada magnitud, en un rango de magnitudes claro. Para esto visualizaremos un histograma:

In [ ]:
#@title Cuenta de estrellas en las dos magnitudes
plt.style.use('dark_background')

cuentas, axs = plt.subplots(1,2, figsize=(10,5))

axs[0].hist(df_stars_mw['Gmag'], bins=50, color='green')
axs[0].set_title("Estrellas en magnitud visible (GAIA)")
axs[0].set_xlabel('Magnitud G')
axs[0].set_ylabel('Número de estrellas')
axs[0].grid(True)

axs[1].hist(df_stars_mw['W1mag'], bins=50, color='red')
axs[1].set_title("Estrellas en magnitud infrarroja (WISE)")
axs[1].set_xlabel('Magnitud I')
axs[1].grid(True)

Ahora calcularemos el índice de color y lo añadiremos al DataFrame como una nueva columna:

In [ ]:
df_stars_mw["color_index"] = df_stars_mw['Gmag'] - df_stars_mw['W1mag']

Crearemos entonces nuestro diagrama color-magnitud para las estrellas de la Vía Láctea en esta región:

In [ ]:
plt.style.use('dark_background')

plt.figure(figsize=(8,8))
plt.scatter(df_stars_mw['Gmag'], df_stars_mw['color_index'])
plt.title("Diagrama color-magnitud de las estrellas en la región observada")
plt.gca().invert_yaxis()
plt.xlabel("Magnitud G (Gmag)")
plt.ylabel("Índice de color G-I")
plt.grid(True)
plt.show()